In [1]:
# Cell 1 — Setup
# Requires:
#   pip install -r requirements.txt
#   python -m spacy download en_core_web_sm
# Launch Jupyter from the repo root OR from the notebooks/ subdirectory.

import sys
from pathlib import Path

# Find repo root: start from cwd and walk up until grammar_parser/ is found.
REPO_ROOT = Path.cwd()
for _ in range(3):
    if (REPO_ROOT / 'grammar_parser').is_dir():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise RuntimeError(
        "Could not locate grammar_parser/ in cwd or its parents.\n"
        "Set REPO_ROOT manually, e.g.:\n"
        "  REPO_ROOT = Path(r'C:/Users/marta/Documents/PROJECT BARCELONA/GRAMMAR')"
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import spacy
from grammar_parser import Group1Parser, Group2Parser, Group3Parser, Group4Parser

nlp = spacy.load('en_core_web_sm')

# resolve=True: filters incompatible-category conflicts on the same span
# using spaCy DEP heuristics. Set resolve=False to see all raw matches.
parser1 = Group1Parser(nlp, resolve=True)   # LEXICAL_TRIGGER  — MODALITY, NEGATION, DISCOURSE MARKERS, CONJUNCTIONS, PREPOSITIONS
parser2 = Group2Parser(nlp, resolve=True)   # NOMINAL_POS      — PRONOUNS, DETERMINERS, NOUNS, ADJECTIVES, ADVERBS
parser3 = Group3Parser(nlp, resolve=True)   # VERBAL_MORPHOLOGY — PAST, PRESENT, FUTURE, PASSIVES, VERBS
parser4 = Group4Parser(nlp, resolve=True)   # SYNTACTIC_STRUCTURE — CLAUSES, REPORTED SPEECH, FOCUS, QUESTIONS

parsers = [parser1, parser2, parser3, parser4]

print(f"Repo root    : {REPO_ROOT}")
print(f"spaCy model  : {nlp.meta['name']} v{nlp.meta['version']}")
print(f"Group1Parser : {len(parser1.structures)} structures  (strategy1 — LEXICAL_TRIGGER)")
print(f"Group2Parser : {len(parser2.structures)} structures  (strategy2 — NOMINAL_POS)")
print(f"Group3Parser : {len(parser3.structures)} structures  (strategy3 — VERBAL_MORPHOLOGY)")
print(f"Group4Parser : {len(parser4.structures)} structures  (strategy4 — SYNTACTIC_STRUCTURE)")
print(f"─" * 52)
total = sum(len(p.structures) for p in parsers)
print(f"Total        : {total} structures across {len(parsers)} parsers")

Repo root    : C:\Users\marta\Documents\PROJECT BARCELONA\GRAMMAR
spaCy model  : core_web_sm v3.8.0
Group1Parser : 30 structures  (strategy1 — LEXICAL_TRIGGER)
Group2Parser : 25 structures  (strategy2 — NOMINAL_POS)
Group3Parser : 53 structures  (strategy3 — VERBAL_MORPHOLOGY)
Group4Parser : 30 structures  (strategy4 — SYNTACTIC_STRUCTURE)
────────────────────────────────────────────────────
Total        : 138 structures across 4 parsers


In [2]:
# Cell 2 — Load lesson sentences
# Reads from validated-sentence-separation.txt (properly segmented sentences).
# Run separate_sentences.py first if the file doesn't exist yet.

LESSON_PATH = REPO_ROOT / 'processed_data' / 'sentences' / 'Student-1' / 'lesson-1' / 'validated-sentence-separation.txt'

def load_sentences(path: Path) -> list:
    with path.open(encoding='utf-8') as fh:
        lines = [line.strip() for line in fh]
    return [line for line in lines if line]

sentences = load_sentences(LESSON_PATH)

print(f"Loaded {len(sentences)} sentences from:")
print(f"  {LESSON_PATH}")
print()
print('First 5 sentences:')
for s in sentences[:5]:
    print(f'  {s!r}')

Loaded 300 sentences from:
  C:\Users\marta\Documents\PROJECT BARCELONA\GRAMMAR\processed_data\sentences\Student-1\lesson-1\validated-sentence-separation.txt

First 5 sentences:
  '\ufeffHello?'
  'Fine.'
  'Thanks.'
  'Marcos, how are you?'
  'Very good.'


In [3]:
# Cell 3 — Pipeline function
# run_pipeline accepts any list of parser objects that implement .parse(doc).
# Adding Group2Parser later: parsers = [parser1, parser2]

def run_pipeline(sentences, nlp, parsers):
    """
    Process all sentences through all parsers.

    Returns
    -------
    list of dict:
        sentence_index : int
        sentence_text  : str
        matches        : list[dict]  — aggregated results from all parsers,
                         each match includes:
                           start_char / end_char         — exact matched span chars
                           context_start_char / context_end_char — wider highlight chars
    """
    results = []
    docs = list(nlp.pipe(sentences))

    for idx, (sentence, doc) in enumerate(zip(sentences, docs)):
        all_matches = []
        for parser in parsers:
            for m in parser.parse(doc):
                # Exact matched span character offsets
                span = doc[m['start_token']:m['end_token']]
                m['start_char'] = span.start_char
                m['end_char']   = span.end_char
                # Wider context window for visual highlighting
                ctx = doc[m['context_start_token']:m['context_end_token']]
                m['context_start_char'] = ctx.start_char
                m['context_end_char']   = ctx.end_char
                all_matches.append(m)
        results.append({
            'sentence_index': idx,
            'sentence_text':  sentence,
            'matches':        all_matches,
        })

    return results

print('run_pipeline defined.')

run_pipeline defined.


In [4]:
# Cell 4 — Run the pipeline

pipeline_results = run_pipeline(sentences, nlp, parsers)

total_matches = sum(len(r['matches']) for r in pipeline_results)
sentences_with_hits = sum(1 for r in pipeline_results if r['matches'])

print('Pipeline complete.')
print(f'  Sentences processed  : {len(pipeline_results)}')
print(f'  Sentences with hits  : {sentences_with_hits}')
print(f'  Total match events   : {total_matches}')

Pipeline complete.
  Sentences processed  : 300
  Sentences with hits  : 149
  Total match events   : 516


In [5]:
# Cell 5 — Visual display (HTML)
#
# Highlighting uses two layers:
#   • Primary   — exact matched span, always visible (DIM_MARK colour, bold)
#   • Secondary — wider syntactic context, shown only on hover (DIM_BG colour, lighter)
#
# Table columns: matched span | dimension | category | guideword | CEFR level | explanation

from IPython.display import HTML, display as _display
from collections import defaultdict
import html as _html

# ── Proficiency Dimensions ────────────────────────────────────────────────────
DIMENSION_MAP = {
    "CLAUSES": "A",  "QUESTIONS": "A",  "REPORTED SPEECH": "A",
    "CONJUNCTIONS": "A",  "DISCOURSE MARKERS": "A",
    "PAST": "B",  "PRESENT": "B",  "FUTURE": "B",
    "PASSIVES": "B",  "VERBS": "B",  "NEGATION": "B",
    "PRONOUNS": "C",  "DETERMINERS": "C",  "NOUNS": "C",
    "ADJECTIVES": "C",  "ADVERBS": "C",  "PREPOSITIONS": "C",
    "MODALITY": "D",  "FOCUS": "D",
}
DIMENSION_LABELS = {
    "A": "Sentence Architecture",
    "B": "Tense & Aspect",
    "C": "Nominal Precision",
    "D": "Modal & Function",
}
# DIM_MARK = strong colour for matched span (primary)
# DIM_BG   = light colour for context on hover (secondary)
DIM_BG   = {"A": "#DBEAFE", "B": "#FEF3C7", "C": "#D1FAE5", "D": "#EDE9FE", "?": "#F3F4F6"}
DIM_TEXT = {"A": "#1D4ED8", "B": "#92400E", "C": "#065F46", "D": "#5B21B6", "?": "#374151"}
DIM_MARK = {"A": "#93C5FD", "B": "#FCD34D", "C": "#6EE7B7", "D": "#C4B5FD", "?": "#D1D5DB"}

CEFR_BG  = {"A1": "#D1FAE5", "A2": "#6EE7B7", "B1": "#FEF3C7",
            "B2": "#FDE68A", "C1": "#FECACA", "C2": "#FCA5A5"}
CEFR_TXT = {"A1": "#065F46", "A2": "#064E3B", "B1": "#78350F",
            "B2": "#92400E", "C1": "#991B1B", "C2": "#7F1D1D"}

# ── CSS injected once (hover effects can't be done with inline styles) ────────
_HIGHLIGHT_CSS = """
<style>
/* Primary highlight: exact matched span — always visible */
.m-hl          { font-weight:700; border-radius:3px; padding:1px 3px; }
.m-hl.m-A      { background:#93C5FD; color:#1D4ED8; }
.m-hl.m-B      { background:#FCD34D; color:#92400E; }
.m-hl.m-C      { background:#6EE7B7; color:#065F46; }
.m-hl.m-D      { background:#C4B5FD; color:#5B21B6; }
.m-hl.m-x      { background:#D1D5DB; color:#374151; }

/* Secondary highlight: context window — visible only on sentence hover */
.c-hl          { border-radius:3px; padding:1px 2px;
                 background:transparent; transition:background 0.15s; }
.sent-block:hover .c-hl.c-A { background:#DBEAFE; color:#1D4ED8; }
.sent-block:hover .c-hl.c-B { background:#FEF3C7; color:#92400E; }
.sent-block:hover .c-hl.c-C { background:#D1FAE5; color:#065F46; }
.sent-block:hover .c-hl.c-D { background:#EDE9FE; color:#5B21B6; }
.sent-block:hover .c-hl.c-x { background:#F3F4F6; color:#374151; }
</style>
"""


def _dim_badge(dim):
    bg = DIM_BG.get(dim, DIM_BG["?"])
    fg = DIM_TEXT.get(dim, DIM_TEXT["?"])
    label = DIMENSION_LABELS.get(dim, dim)
    return (f'<span style="background:{bg};color:{fg};padding:2px 8px;'
            f'border-radius:10px;font-size:11px;font-weight:600;white-space:nowrap;">'
            f'{dim} &middot; {label}</span>')


def _cefr_badge(level):
    bg = CEFR_BG.get(level, "#F3F4F6")
    fg = CEFR_TXT.get(level, "#374151")
    return (f'<span style="background:{bg};color:{fg};padding:2px 7px;'
            f'border-radius:8px;font-size:11px;font-weight:700;opacity:0.5;">'
            f'{level}</span>')


def _cefr_badge_assigned(level):
    bg = CEFR_BG.get(level, "#F3F4F6")
    fg = CEFR_TXT.get(level, "#374151")
    return (
        f'<span style="background:{bg};color:{fg};padding:2px 7px;'
        f'border-radius:8px;font-size:11px;font-weight:700;'
        f'border:2px solid {fg};text-decoration:underline;">'
        f'{level}</span>'
        f'<span style="font-size:10px;color:#6B7280;margin-left:5px;'
        f'font-style:italic;">assigned</span>'
    )


def _highlight_sentence(text, match_spans, context_spans):
    """
    Two-tier highlighting via CSS classes (hover handled in _HIGHLIGHT_CSS).

    match_spans   : list of (s, e, dim) — always shown with DIM_MARK colour
    context_spans : list of (s, e, dim) — shown on hover with DIM_BG colour

    Algorithm: assign each character position a tier ('match' > 'ctx' > None),
    then emit spans for consecutive runs of the same tier+dim.
    """
    n    = len(text)
    tier = [None] * n  # each slot: None | ('ctx', dim) | ('match', dim)

    # Lower priority first
    for s, e, dim in context_spans:
        css = dim if dim in "ABCD" else "x"
        for i in range(max(0, s), min(e, n)):
            if tier[i] is None:
                tier[i] = ('ctx', css)

    # Higher priority: matched span overwrites context
    for s, e, dim in match_spans:
        css = dim if dim in "ABCD" else "x"
        for i in range(max(0, s), min(e, n)):
            tier[i] = ('match', css)

    parts = []
    i = 0
    while i < n:
        t = tier[i]
        if t is None:
            j = i
            while j < n and tier[j] is None:
                j += 1
            parts.append(_html.escape(text[i:j]))
            i = j
        else:
            kind, css = t
            j = i
            while j < n and tier[j] == t:
                j += 1
            chunk = _html.escape(text[i:j])
            cls = f"m-hl m-{css}" if kind == "match" else f"c-hl c-{css}"
            tag = "mark" if kind == "match" else "span"
            parts.append(f'<{tag} class="{cls}">{chunk}</{tag}>')
            i = j

    return ''.join(parts)


def display_results_html(pipeline_results, max_sentences=15, max_rows_per_cat=5):
    blocks = []
    shown  = 0

    for result in pipeline_results:
        matches = result['matches']
        if not matches:
            continue
        if shown >= max_sentences:
            remaining = sum(1 for r in pipeline_results if r['matches']) - shown
            blocks.append(
                f'<p style="color:#6B7280;font-style:italic;margin-top:8px;">'
                f'&#x2026; {remaining} more sentence(s) with matches not shown. '
                f'Increase <code>max_sentences</code> to see more.</p>'
            )
            break
        shown += 1

        idx           = result['sentence_index']
        sentence_text = result['sentence_text']

        # ── Build the two span sets for highlighting ──────────────────────────
        match_spans_seen   = {}  # (s,e) → dim
        context_spans_seen = {}  # (s,e) → dim
        for m in matches:
            dim = DIMENSION_MAP.get(m['category'], '?')
            mk = (m['start_char'], m['end_char'])
            ck = (m['context_start_char'], m['context_end_char'])
            if mk not in match_spans_seen:
                match_spans_seen[mk] = dim
            if ck not in context_spans_seen:
                context_spans_seen[ck] = dim

        highlighted = _highlight_sentence(
            sentence_text,
            [(s, e, d) for (s, e), d in match_spans_seen.items()],
            [(s, e, d) for (s, e), d in context_spans_seen.items()],
        )

        # ── Table: grouped by exact matched span + category ───────────────────
        span_cat = defaultdict(lambda: defaultdict(list))
        for m in matches:
            span_cat[(m['start_char'], m['end_char'])][m['category']].append(m)

        rows = []
        for (s, e), cat_dict in sorted(span_cat.items()):
            span_raw = sentence_text[s:e]
            first_span_in_group = True
            for cat, cat_matches in sorted(cat_dict.items()):
                dim = DIMENSION_MAP.get(cat, '?')
                sorted_m = sorted(cat_matches, key=lambda x: x['lowest_level_numeric'], reverse=True)
                show     = sorted_m[:max_rows_per_cat]
                extra    = len(sorted_m) - max_rows_per_cat

                for i, m in enumerate(show):
                    is_assigned = (i == 0)

                    span_cell = ''
                    if first_span_in_group and i == 0:
                        span_cell = (
                            f'<code style="background:#F3F4F6;padding:2px 6px;'
                            f'border-radius:4px;font-size:12px;">'
                            f'&ldquo;{_html.escape(span_raw)}&rdquo;</code>'
                        )

                    dim_cell = _dim_badge(dim) if i == 0 else ''
                    cat_cell = (
                        f'<span style="font-size:11px;font-weight:600;color:#374151;">'
                        f'{_html.escape(cat)}</span>'
                    ) if i == 0 else ''

                    gw_style = 'font-size:12px;color:#111827;font-weight:600;' if is_assigned else 'font-size:12px;color:#9CA3AF;'
                    gw_cell  = f'<span style="{gw_style}">{_html.escape(m["guideword"])}</span>'

                    level_cell = (
                        _cefr_badge_assigned(m["lowest_level"])
                        if is_assigned
                        else _cefr_badge(m["lowest_level"])
                    )

                    expl = m.get("explanation", "")
                    expl_cell = (
                        f'<span style="font-size:11px;color:#6B7280;font-style:italic;">'
                        f'{_html.escape(expl)}</span>'
                        if is_assigned and expl else ''
                    )

                    row_bg = 'background:#FAFFFE;' if is_assigned else ''
                    rows.append(
                        f'<tr style="border-bottom:1px solid #F3F4F6;{row_bg}">'
                        f'<td style="padding:5px 8px;vertical-align:top;">{span_cell}</td>'
                        f'<td style="padding:5px 8px;vertical-align:top;">{dim_cell}</td>'
                        f'<td style="padding:5px 8px;vertical-align:top;">{cat_cell}</td>'
                        f'<td style="padding:5px 8px;vertical-align:top;">{gw_cell}</td>'
                        f'<td style="padding:5px 8px;vertical-align:top;white-space:nowrap;">{level_cell}</td>'
                        f'<td style="padding:5px 8px;vertical-align:top;max-width:280px;">{expl_cell}</td>'
                        f'</tr>'
                    )
                    first_span_in_group = False

                if extra > 0:
                    rows.append(
                        f'<tr><td colspan="6" style="padding:2px 8px 6px;font-size:11px;'
                        f'color:#9CA3AF;font-style:italic;">'
                        f'&nbsp;&nbsp;&nbsp;+ {extra} more {_html.escape(cat)} guideword(s) not shown'
                        f'</td></tr>'
                    )

        # sent-block class enables the :hover CSS selector in _HIGHLIGHT_CSS
        blocks.append(f'''
        <div class="sent-block" style="margin:14px 0;padding:14px 16px;border:1px solid #E5E7EB;
                    border-radius:8px;font-family:system-ui,-apple-system,sans-serif;
                    background:#FAFAFA;">
          <div style="font-size:11px;color:#9CA3AF;margin-bottom:6px;
                      letter-spacing:.04em;">SENTENCE {idx:03d}</div>
          <div style="font-size:14px;line-height:1.8;margin-bottom:12px;
                      padding:8px 12px;background:#FFFFFF;border-radius:6px;
                      border-left:3px solid #D1D5DB;">
            {highlighted}
          </div>
          <table style="width:100%;border-collapse:collapse;font-size:12px;">
            <thead>
              <tr style="background:#F9FAFB;border-bottom:2px solid #E5E7EB;">
                <th style="padding:6px 8px;text-align:left;color:#6B7280;font-size:10px;text-transform:uppercase;letter-spacing:.06em;">Matched span</th>
                <th style="padding:6px 8px;text-align:left;color:#6B7280;font-size:10px;text-transform:uppercase;letter-spacing:.06em;">Dimension</th>
                <th style="padding:6px 8px;text-align:left;color:#6B7280;font-size:10px;text-transform:uppercase;letter-spacing:.06em;">Category</th>
                <th style="padding:6px 8px;text-align:left;color:#6B7280;font-size:10px;text-transform:uppercase;letter-spacing:.06em;">Guideword</th>
                <th style="padding:6px 8px;text-align:left;color:#6B7280;font-size:10px;text-transform:uppercase;letter-spacing:.06em;">Level</th>
                <th style="padding:6px 8px;text-align:left;color:#6B7280;font-size:10px;text-transform:uppercase;letter-spacing:.06em;">Explanation</th>
              </tr>
            </thead>
            <tbody>{''.join(rows)}</tbody>
          </table>
        </div>''')

    # ── Legend + CSS injection ────────────────────────────────────────────────
    legend_items = '&nbsp;&nbsp;'.join(_dim_badge(d) for d in 'ABCD')
    legend = (
        f'{_HIGHLIGHT_CSS}'
        f'<div style="margin-bottom:12px;padding:10px 16px;background:#F9FAFB;'
        f'border-radius:8px;font-family:system-ui,sans-serif;'
        f'border:1px solid #E5E7EB;">'
        f'<span style="font-size:11px;font-weight:600;color:#374151;'
        f'margin-right:12px;">Proficiency Dimensions</span>'
        f'{legend_items}'
        f'<span style="font-size:11px;color:#9CA3AF;margin-left:20px;">'
        f'Bold highlight = matched span &nbsp;&middot;&nbsp; '
        f'Hover sentence for full syntactic context &nbsp;&middot;&nbsp; '
        f'Underlined badge = assigned level</span>'
        f'</div>'
    )

    _display(HTML(legend + '\n'.join(blocks)))


display_results_html(pipeline_results)

Matched span,Dimension,Category,Guideword,Level,Explanation
“are”,A · Sentence Architecture,QUESTIONS,FORM: AUXILIARY 'BE',A2assigned,"Yes/no question with 'be' — question formed by moving 'is/are/was/were' before the subject (e.g. 'Is she coming?', 'Were they late?')."
“you?”,A · Sentence Architecture,QUESTIONS,FORM: QUESTION TAGS,A2assigned,"Question tag — short question added at the end of a statement to seek confirmation or agreement (e.g. 'It's nice, isn't it?', 'You know him, don't you?')."
Matched span,Dimension,Category,Guideword,Level,Explanation
“Very good”,C · Nominal Precision,ADJECTIVES,FORM: WITH DEGREE ADVERBS,A2assigned,"Adjective with degree adverb — intensifier or softener modifying an adjective (e.g. 'very big', 'quite interesting', 'rather cold')."
Matched span,Dimension,Category,Guideword,Level,Explanation
“teaches”,A · Sentence Architecture,CLAUSES,FORM: WITH 'WHOSE NAME',B1assigned,"Relative clause with 'whose' — 'whose' introduces a clause showing possession or close relationship (e.g. 'the woman whose bag was stolen', 'the doctor whose name I forgot')."
,,,"FORM: DEFINING, SUBJECT, WITH 'WHO'",A2,
“very kind”,C · Nominal Precision,ADJECTIVES,FORM: WITH DEGREE ADVERBS,A2assigned,"Adjective with degree adverb — intensifier or softener modifying an adjective (e.g. 'very big', 'quite interesting', 'rather cold')."
Matched span,Dimension,Category,Guideword,Level,Explanation
“loves”,A · Sentence Architecture,CLAUSES,FORM: WITH 'WHOSE NAME',B1assigned,"Relative clause with 'whose' — 'whose' introduces a clause showing possession or close relationship (e.g. 'the woman whose bag was stolen', 'the doctor whose name I forgot')."


In [6]:
# Cell 6 — Summary by Proficiency Dimension + CEFR level
#
# Counts are based on unique (span, category) groups, each contributing once
# at its ASSIGNED level (highest lowest_level_numeric in the group). This
# credits the student for the most advanced grammatical use they demonstrate,
# avoiding undervaluation from generic low-level patterns that fire alongside
# more specific higher-level ones.

def display_summary_html(pipeline_results):
    from collections import Counter

    dim_counts   = Counter()
    level_counts = Counter()
    cat_counts   = Counter()

    sentences_hit = 0
    total_groups  = 0

    for result in pipeline_results:
        matches = result['matches']
        if matches:
            sentences_hit += 1

        # Deduplicate to one match per (start_token, end_token, category),
        # keeping the HIGHEST-level match (assigned level = ceiling of what
        # the student demonstrates with that span).
        groups: dict[tuple, dict] = {}
        for m in matches:
            key = (m['start_token'], m['end_token'], m['category'])
            if key not in groups or m['lowest_level_numeric'] > groups[key]['lowest_level_numeric']:
                groups[key] = m

        for m in groups.values():
            dim = DIMENSION_MAP.get(m['category'], '?')
            dim_counts[dim]                += 1
            level_counts[m['lowest_level']] += 1
            cat_counts[m['category']]        += 1
            total_groups                     += 1

    if total_groups == 0:
        _display(HTML('<p>No matches found.</p>'))
        return

    # ── CEFR bar chart ────────────────────────────────────────────────────────
    max_lv = max(level_counts.values())
    lv_rows = []
    for lv in ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']:
        n   = level_counts.get(lv, 0)
        pct = n / total_groups * 100
        w   = n / max_lv * 240
        bg  = CEFR_BG.get(lv, '#E5E7EB')
        lv_rows.append(
            f'<tr>'
            f'<td style="padding:4px 10px 4px 0;vertical-align:middle;">{_cefr_badge_assigned(lv)}</td>'
            f'<td style="padding:4px 4px;vertical-align:middle;">'
            f'<div style="background:{bg};height:16px;width:{w:.0f}px;'
            f'border-radius:3px;display:inline-block;min-width:4px;"></div></td>'
            f'<td style="padding:4px 8px;font-size:12px;color:#374151;'
            f'white-space:nowrap;">{n:,} &nbsp;<span style="color:#9CA3AF;">({pct:.1f}%)</span></td>'
            f'</tr>'
        )

    # ── Dimension bar chart ───────────────────────────────────────────────────
    max_dm = max(dim_counts.values()) if dim_counts else 1
    dm_rows = []
    for dim in 'ABCD':
        n   = dim_counts.get(dim, 0)
        pct = n / total_groups * 100
        w   = n / max_dm * 240
        mk  = DIM_MARK.get(dim, '#D1D5DB')
        dm_rows.append(
            f'<tr>'
            f'<td style="padding:4px 10px 4px 0;vertical-align:middle;">{_dim_badge(dim)}</td>'
            f'<td style="padding:4px 4px;vertical-align:middle;">'
            f'<div style="background:{mk};height:16px;width:{w:.0f}px;'
            f'border-radius:3px;display:inline-block;min-width:4px;"></div></td>'
            f'<td style="padding:4px 8px;font-size:12px;color:#374151;'
            f'white-space:nowrap;">{n:,} &nbsp;<span style="color:#9CA3AF;">({pct:.1f}%)</span></td>'
            f'</tr>'
        )

    # ── Category breakdown (top 10) ───────────────────────────────────────────
    cat_rows = []
    for cat, n in cat_counts.most_common(10):
        dim  = DIMENSION_MAP.get(cat, '?')
        pct  = n / total_groups * 100
        mark = DIM_MARK.get(dim, '#D1D5DB')
        w    = n / cat_counts.most_common(1)[0][1] * 180
        cat_rows.append(
            f'<tr style="border-bottom:1px solid #F3F4F6;">'
            f'<td style="padding:4px 8px;font-size:12px;font-weight:600;color:#1F2937;">'
            f'{_html.escape(cat)}</td>'
            f'<td style="padding:4px 8px;">{_dim_badge(dim)}</td>'
            f'<td style="padding:4px 8px;">'
            f'<div style="background:{mark};height:14px;width:{w:.0f}px;'
            f'border-radius:3px;display:inline-block;min-width:4px;"></div></td>'
            f'<td style="padding:4px 8px;font-size:12px;color:#374151;'
            f'white-space:nowrap;">{n:,} &nbsp;<span style="color:#9CA3AF;">({pct:.1f}%)</span></td>'
            f'</tr>'
        )

    html = f'''
    <div style="font-family:system-ui,-apple-system,sans-serif;padding:16px 0;">
      <h3 style="color:#1F2937;margin:0 0 6px 0;font-size:16px;">Results Summary</h3>
      <p style="font-size:13px;color:#6B7280;margin:0 0 20px 0;">
        {len(pipeline_results)} sentences &nbsp;&middot;&nbsp;
        {sentences_hit} with matches &nbsp;&middot;&nbsp;
        {total_groups:,} grammar events (unique span &times; category, at assigned level)
      </p>

      <div style="display:flex;gap:40px;flex-wrap:wrap;align-items:flex-start;">

        <div>
          <div style="font-size:11px;font-weight:600;color:#6B7280;
                      text-transform:uppercase;letter-spacing:.06em;
                      margin-bottom:8px;">CEFR Level (assigned)</div>
          <table style="border-collapse:collapse;">{''.join(lv_rows)}</table>
        </div>

        <div>
          <div style="font-size:11px;font-weight:600;color:#6B7280;
                      text-transform:uppercase;letter-spacing:.06em;
                      margin-bottom:8px;">Proficiency Dimension</div>
          <table style="border-collapse:collapse;">{''.join(dm_rows)}</table>
        </div>

        <div>
          <div style="font-size:11px;font-weight:600;color:#6B7280;
                      text-transform:uppercase;letter-spacing:.06em;
                      margin-bottom:8px;">Top Categories</div>
          <table style="border-collapse:collapse;">
            <thead>
              <tr style="background:#F9FAFB;border-bottom:2px solid #E5E7EB;">
                <th style="padding:6px 8px;text-align:left;font-size:10px;color:#9CA3AF;">Category</th>
                <th style="padding:6px 8px;text-align:left;font-size:10px;color:#9CA3AF;">Dimension</th>
                <th colspan="2" style="padding:6px 8px;text-align:left;font-size:10px;color:#9CA3AF;">Count</th>
              </tr>
            </thead>
            <tbody>{''.join(cat_rows)}</tbody>
          </table>
        </div>

      </div>
    </div>
    '''
    _display(HTML(html))


display_summary_html(pipeline_results)

A1assigned,,0 (0.0%)
A2assigned,,172 (64.2%)
B1assigned,,53 (19.8%)
B2assigned,,30 (11.2%)
C1assigned,,0 (0.0%)
C2assigned,,13 (4.9%)
A · Sentence Architecture,,102 (38.1%)
B · Tense & Aspect,,111 (41.4%)
C · Nominal Precision,,42 (15.7%)
D · Modal & Function,,13 (4.9%)
Category,Dimension,Count


In [7]:
# Cell 7 — Sentences with highest assigned CEFR level
#
# Shows sentences where at least one (span × category) group reaches
# the target level or above. Change MIN_LEVEL to filter differently.
# Uses the same display_results_html function from Cell 5.

MIN_LEVEL = "C1"   # "B2", "C1" or "C2"

CEFR_ORDER = {"A1": 1, "A2": 2, "B1": 3, "B2": 4, "C1": 5, "C2": 6}
min_numeric = CEFR_ORDER[MIN_LEVEL]


def _assigned_level_numeric(matches):
    """Return the highest assigned level across all (span, category) groups."""
    groups: dict[tuple, dict] = {}
    for m in matches:
        key = (m['start_token'], m['end_token'], m['category'])
        if key not in groups or m['lowest_level_numeric'] > groups[key]['lowest_level_numeric']:
            groups[key] = m
    if not groups:
        return 0
    return max(m['lowest_level_numeric'] for m in groups.values())


high_level_results = [
    r for r in pipeline_results
    if _assigned_level_numeric(r['matches']) >= min_numeric
]

print(f"Sentences with at least one match at {MIN_LEVEL}+: {len(high_level_results)}")
display_results_html(high_level_results, max_sentences=30)

Sentences with at least one match at C1+: 12


Matched span,Dimension,Category,Guideword,Level,Explanation
“His”,C · Nominal Precision,PRONOUNS,FORM: 'HIS',C2assigned,'His' as standalone pronoun — masculine possessive pronoun used alone as a noun phrase (e.g. 'Is that jacket his?').
,,,"FORM: OF 'THEIRS', 'HERS', 'HIS'",C2,
“are nice”,B · Tense & Aspect,VERBS,FORM: LINKING + COMPLEMENT,A2assigned,"Linking verb with complement — verb like 'seem', 'become', 'appear' connecting the subject to a description (e.g. 'He seems tired')."
,,,FORM: LINKING VERBS + ADJECTIVE,A2,
Matched span,Dimension,Category,Guideword,Level,Explanation
“His”,C · Nominal Precision,PRONOUNS,FORM: 'HIS',C2assigned,'His' as standalone pronoun — masculine possessive pronoun used alone as a noun phrase (e.g. 'Is that jacket his?').
,,,"FORM: OF 'THEIRS', 'HERS', 'HIS'",C2,
“is funny”,B · Tense & Aspect,VERBS,FORM: LINKING + COMPLEMENT,A2assigned,"Linking verb with complement — verb like 'seem', 'become', 'appear' connecting the subject to a description (e.g. 'He seems tired')."
,,,FORM: LINKING VERBS + ADJECTIVE,A2,
Matched span,Dimension,Category,Guideword,Level,Explanation
